In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
import json, torch, os # os modülünü ekledik
from torch.utils.data import Dataset

# Checkpoint'ten devam et
checkpoint = '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657'

# Hata kontrolü ekle: Dizinin varlığını ve içerdiği dosyaları kontrol et
if not os.path.exists(checkpoint):
    print(f"Hata: Belirtilen checkpoint dizini bulunamadı: {checkpoint}. Lütfen yolun doğru olduğundan ve dizinin mevcut olduğundan emin olun.")
    print("\nCheckpoint dizininin üst klasöründeki mevcut dizinleri listeliyorum:")
    parent_dir = os.path.dirname(checkpoint)
    if os.path.exists(parent_dir) and os.path.isdir(parent_dir):
        print(os.listdir(parent_dir))
    else:
        print(f"Uyarı: '{parent_dir}' üst dizini de bulunamadı.")
    raise FileNotFoundError(f"Hata: Belirtilen checkpoint dizini bulunamadı: {checkpoint}. Lütfen yolun doğru olduğundan ve dizinin mevcut olduğundan emin olun.")

if not os.path.isdir(checkpoint):
    raise NotADirectoryError(f"Hata: Belirtilen checkpoint yolu bir dizin değil: {checkpoint}. Lütfen bir dizin yolu sağlayın.")

# Gerekli dosyaların varlığını kontrol edelim
# Bu dosyalar modelin ve tokenizer'ın düzgün yüklenmesi için kritik öneme sahiptir.
required_model_files = ['config.json', 'pytorch_model.bin'] # veya modelinize özgü diğer dosyalar
required_tokenizer_files = ['tokenizer_config.json', 'vocab.json', 'merges.txt', 'spiece.model'] # tokenizer tipine göre değişebilir

for fname in required_model_files:
    if not os.path.exists(os.path.join(checkpoint, fname)):
        print(f"Uyarı: '{checkpoint}' dizininde '{fname}' dosyası bulunamadı. Bu durum modelin yüklenmesinde sorunlara neden olabilir.")

for fname in required_tokenizer_files:
    if not os.path.exists(os.path.join(checkpoint, fname)):
        print(f"Uyarı: '{checkpoint}' dizininde '{fname}' dosyası bulunamadı. Bu durum tokenizer'ın yüklenmesinde sorunlara neden olabilir.")


model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Veriyi yükle — sadece 10.000 örnek
data = []
with open('/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl') as f:
    for i, line in enumerate(f):
        if i >= 10000:
            break
        data.append(json.loads(line))

print(f"Veri yüklendi: {len(data)} örnek")

class CodeReviewDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        inp = self.tokenizer(item['input'], max_length=512, truncation=True, padding='max_length', return_tensors='pt')
        tgt = self.tokenizer(item['msg'], max_length=128, truncation=True, padding='max_length', return_tensors='pt')
        labels = tgt['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'input_ids': inp['input_ids'].squeeze(), 'attention_mask': inp['attention_mask'].squeeze(), 'labels': labels}

dataset = CodeReviewDataset(data, tokenizer)
print(f"Dataset hazır: {len(dataset)} örnek")

# 10 epoch eğit
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
print("Eğitim başlıyor... (3-4 saat sürecek, Colab'ı kapatma!)")
trainer.train()

# Kaydet
model.save_pretrained('/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/')
tokenizer.save_pretrained('/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/')
print("✅ Eğitim tamamlandı ve kaydedildi!")

Uyarı: '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657' dizininde 'pytorch_model.bin' dosyası bulunamadı. Bu durum modelin yüklenmesinde sorunlara neden olabilir.
Uyarı: '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657' dizininde 'tokenizer_config.json' dosyası bulunamadı. Bu durum tokenizer'ın yüklenmesinde sorunlara neden olabilir.
Uyarı: '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657' dizininde 'vocab.json' dosyası bulunamadı. Bu durum tokenizer'ın yüklenmesinde sorunlara neden olabilir.
Uyarı: '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657' dizininde 'merges.txt' dosyası bulunamadı. Bu durum tokenizer'ın yüklenmesinde sorunlara neden olabilir.
Uyarı: '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657' dizininde 'spiece.model' dosyası bulunamadı. Bu durum tokenizer'ın yüklenmesinde sorunlara neden olabilir.


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Veri yüklendi: 10000 örnek
Dataset hazır: 10000 örnek
Eğitim başlıyor... (3-4 saat sürecek, Colab'ı kapatma!)


Step,Training Loss
500,0.270852
1000,0.114122
1500,0.112328
2000,0.113365
2500,0.110066
3000,0.111326
3500,0.108980
4000,0.111111
4500,0.109008
5000,0.108224


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Eğitim tamamlandı ve kaydedildi!
